In [ ]:
%cd ../../..

In [ ]:
from itertools import zip_longest, combinations
from pathlib import Path
import sys

import polars as pl
import networkx as nx
from loguru import logger

In [ ]:
logger.remove()
logger.add(sys.stdout, level="DEBUG")

# Read data

In [ ]:
paths = Path("data/inter/inter_dim_meals").glob("*.xlsx")

schema = {
    "meal_id": pl.Int32,
    "names": pl.String,
    "restaurants": pl.String,
    "meal_type": pl.String,
    "schoolyear": pl.String,
    "attributes": pl.String,
    "co2": pl.Float32
}

df_list = []
for path in paths:
    df = pl.read_excel(path, schema_overrides=schema)
    df = df.with_columns(pl.lit(path.stem).alias('src'))
    df_list.append(df)

meals_inter = pl.concat(df_list)
meals_inter.head()

In [ ]:
meals_inter.filter(
    (1 == 1)
    & (pl.col('names') == pl.lit("Paneroitu lohipihvi, sitruunak"))
)

In [ ]:
path = "data/processed/dim_meal_types.xlsx"
dim_meal_types = pl.read_excel(path)
dim_meal_types.head()

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pl.read_excel(path)
dim_restaurants.head()

In [ ]:
# This table is dedicated to solve the problem: same meal has multiple meal_types (from multiple sources)
dim_src_rank = pl.from_dicts([
    {'src': "pos_Nov24-Mar25",  'priority': 1},
    {'src': "pos_Jan23-Oct24",  'priority': 2},
    {'src': "menus_week1-6",    'priority': 3},
    {'src': "menus_baguettes",  'priority': 4},
    {'src': "menus_meals",      'priority': 5},

])

# Process

In [ ]:
meals = (
    meals_inter
    .with_columns(
        pl.col('attributes').str.split('|')
    )
    
    .join(dim_meal_types, left_on='meal_type', right_on='meal_type_en', how='left')
    # .filter(
    #     (1 == 1)
    #     & (pl.col('meal_type_id').is_null())
    #     & (pl.col('meal_type').is_not_null())
    # )
    .with_columns(pl.col('meal_type_id').alias('meal_type'))
    .drop('meal_type_id', 'meal_type_right')

    .with_columns(
        pl.col('restaurants').replace({'not_phy': "che|exa|vik"}).str.split("|")
    )
    .explode('restaurants')
    .join(dim_restaurants, left_on='restaurants', right_on='restaurant_short', how='left')
    # .filter(
    #     (1 == 1)
    #     & (pl.col('restaurant_id').is_null())
    #     & (pl.col('restaurants').is_not_null())
    # )
    .with_columns(
        pl.col('restaurant_id').alias('restaurants')
    )
    .drop('restaurant', 'restaurant_id')


    .with_row_index()
)

meals.head()

# Build `dim_meals`

Idea: Consider the entries are nodes in graph. Two nodes are connected if the corresponding entries have something in common. Hence, the group of similar entries is the connected subgraphs.

There are 3 cases that two entries are similar:
- two entries have same `meal_id`
- two entries have same `names`
- two entries have closest valid Hamming distance of `names` (valid Hamming distance is distance less than or equal `THRESH = 0.1`)

## Group "similar" entries

In [ ]:
group1 = (
    meals
    .filter(pl.col('meal_id').is_not_null())
    .group_by(
        pl.col('meal_id')
    )
    .agg(
        pl.concat_list('index').flatten()
    )
    .filter(pl.col('index').list.len() > 1)
    ['index']
    .to_list()
)

group1[:5]

In [ ]:
group2 = (
    meals
    .group_by('names')
    .agg(
        pl.concat_list('index').flatten()
    )
    .filter(pl.col('index').list.len() > 1)
    ['index']
    .to_list()
)

group2[:5]

In [ ]:
THRES = 0.1
def hamming_distance(record) -> float:
    s1 = record['names']
    s2 = record['names_right']
    min_len = min(len(s1), len(s2))
    dist = sum(c1 != c2 for c1, c2 in zip_longest(s1, s2))  * 1.0 / min_len

    return dist


df = meals.select('index', 'meal_id', 'names')
distances = (
    df
    .join(df, how='cross')
    .filter(pl.col('index_right') != pl.col('index'))
    .with_columns(
        pl.when(pl.col('meal_id') == pl.col('meal_id_right'))
        .then(0.)
        .otherwise(pl.struct('names', 'names_right').map_elements(hamming_distance, return_dtype=pl.Float32))
        .alias('dist')
    )

)


group3 = (
    distances
    .filter(
        (pl.col('dist') > 0)
        & (pl.col('dist') <= THRES)
    )
    .with_columns(
        pl.col('dist').rank(method="dense", descending=False).over(['index'], order_by='dist').alias('rank')
    )
    .filter(pl.col('rank') == 1)
    .with_columns(
        pl.concat_list('index', 'index_right').alias('index')
    )
    .filter(pl.col('index').list.len() > 1)
    ['index']
    .to_list()
)

group3[:5]

In [ ]:
# Create list of nodes
nodes = meals['index'].to_list()

# Create list of edges
groups = [*group1, *group2, *group3]
edges = []
for group in groups:
    for v1, v2 in combinations(group, 2):
        edges.append((v1, v2))
        edges.append((v2, v1))

G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

subgraphs_list = []
for idx_subgraph, component in enumerate(nx.connected_components(G)):
    for node in component:
        subgraphs_list.append({'index': node, 'subgraph': idx_subgraph})

subgraphs = pl.from_dicts(subgraphs_list)
subgraphs.head()

In [ ]:
cols = ['meal_id', 'names', 'schoolyear', 'attributes', 'co2', 'meal_type', 'restaurants', 'src']
meals = (
    meals
    .join(subgraphs, on='index', how='left')

    # Supplement source priority info for meal_type
    .join(dim_src_rank, on='src', how='left')
    .with_columns(
        pl.struct('priority', 'meal_type').alias('meal_type')
    )
    .group_by('subgraph')
    .agg(
        *[pl.concat_list(col).flatten().unique().drop_nulls() for col in cols]
    )
    # .filter(pl.col('meal_type').list.len() > 1)
    .with_columns(
        pl.col('schoolyear').list.max(),
        pl.col('co2').list.max(),
    )
    .rename({'subgraph': 'id', 'meal_id': 'meal_ids'})

    # .filter(pl.col('id') == 320)
)

meals.head()

## Select single meal_type

In [ ]:
meal_types = (
    meals
    .select('id', 'meal_type')
    .explode('meal_type')
    .unnest('meal_type')
    .filter(pl.col('meal_type').is_not_null())
    .with_columns(pl.col('priority').rank('ordinal').over('id').alias('rank'))
    .filter(pl.col('rank') == 1)
    .select('id', 'meal_type')
)

meals = (
    meals
    .drop('meal_type')
    .join(meal_types, on='id', how='left')
)

meals.head()

## Supplement `meal_ids` for meals having no meal_id beforehand

In [ ]:
ID_START = 90_000_000

meals = (
    meals
    .sort('id')
    .with_row_index('count')
    .with_columns(
        pl.when(pl.col('meal_ids').list.len() == 0)
        .then(pl.concat_list(ID_START + pl.col('count'), pl.col('meal_ids')))
        .otherwise(pl.col('meal_ids'))
        .alias('meal_codes')
    )
    .drop('count')
)

## Sanity checks

In [ ]:
assert meals.select(pl.col('id').unique()).count().item() == meals.shape[0]
assert meals.filter(pl.col('meal_codes').list.len() < 1).shape[0] == 0

# Save

In [ ]:
cols = ['id', 'meal_codes', 'names', 'restaurants', 'meal_type', 'schoolyear', 'attributes', 'co2', 'src']
path = "data/processed/dim_meals.parquet"
meals.select(*cols).write_parquet(path)